In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber',
                 'DailyRate', 'HourlyRate', 'MonthlyRate']
data.drop(columns=cols_to_drop, inplace=True)

data["Attrition"] = data["Attrition"].map({"Yes": 1, "No": 0})

X = data.drop(columns=["Attrition"])
y = data["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

colonne_categoriche = X_train.select_dtypes(include="object").columns.tolist()

X_train_enc = pd.get_dummies(X_train, columns=colonne_categoriche, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=colonne_categoriche, drop_first=True)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

In [4]:
train_df = X_train_enc.copy()
train_df.insert(0, "Attrition", y_train.values)

test_df = X_test_enc.copy()
test_df.insert(0, "Attrition", y_test.values)

In [5]:
import boto3, sagemaker, io, os

sess = sagemaker.Session()
bucket='c213913a5405381l15843604t1w180050224022-labbucket-sfwgpzgbgknq'
prefix = 'hr-attrition'

s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, index=False)   # header остаётся, index=False
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv('train.csv', 'train', train_df)
upload_s3_csv('test.csv', 'test', test_df)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [11]:
from sagemaker.sklearn.estimator import SKLearn

train_channel = sagemaker.inputs.TrainingInput(
    f"s3://{bucket}/{prefix}/train/", content_type='text/csv')
test_channel = sagemaker.inputs.TrainingInput(
    f"s3://{bucket}/{prefix}/test/", content_type='text/csv')

output_path = f"s3://{bucket}/{prefix}/output/"

logreg_estimator = SKLearn(
    entry_point='train.py',
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type='ml.m4.xlarge',
    framework_version='1.2-1',
    output_path=output_path,
    base_job_name='hr-attrition-logreg',
    hyperparameters={
        'model-type': 'logreg',
        'max-iter': 1000,
        'threshold': 0.6,  
    },
)

logreg_estimator.fit({'train': train_channel, 'test': test_channel}, logs=False)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


INFO:sagemaker:Creating training-job with name: hr-attrition-logreg-2026-07-12-21-20-57-894


Using provided s3_resource

2026-07-12 21:20:59 Starting - Starting the training job..
2026-07-12 21:21:13 Starting - Preparing the instances for training...
2026-07-12 21:21:38 Downloading - Downloading input data..................
2026-07-12 21:23:13 Downloading - Downloading the training image
2026-07-12 21:23:14 Training - Training image download completed. Training in progress....
2026-07-12 21:23:34 Uploading - Uploading generated training model.
2026-07-12 21:23:47 Completed - Training job completed


In [13]:
rf_estimator = SKLearn(
    entry_point='train.py',
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type='ml.m4.xlarge',
    framework_version='1.2-1',
    output_path=output_path,
    base_job_name='hr-attrition-rf',
    hyperparameters={
        'model-type': 'rf',
        'n-estimators': 300,
        'threshold': 0.2,
    },
)

rf_estimator.fit({'train': train_channel, 'test': test_channel}, logs=False)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


INFO:sagemaker:Creating training-job with name: hr-attrition-rf-2026-07-12-21-26-41-879


Using provided s3_resource

2026-07-12 21:26:43 Starting - Starting the training job.
2026-07-12 21:26:57 Starting - Preparing the instances for training....
2026-07-12 21:27:20 Downloading - Downloading input data.....
2026-07-12 21:27:50 Downloading - Downloading the training image..........
2026-07-12 21:28:46 Training - Training image download completed. Training in progress.....
2026-07-12 21:29:11 Uploading - Uploading generated training model..
2026-07-12 21:29:24 Completed - Training job completed
